In [ ]:
import numpy as np
import tensorflow as tf
import keras
import matplotlib.pyplot as plt

import sys

sys.path.append("../")

In [ ]:
import numpy as np

seq_length = 60
dim = 2
noise = 0


def sample_circle(n, max_n=seq_length, dim=2, noise=0.05):
    points = np.full((max_n, dim), -1, dtype=np.float32)
    sample_points = np.random.randn(n, dim)
    sample_points /= np.linalg.norm(sample_points, axis=1, keepdims=True)
    r = np.random.uniform(0.5, 1, size = (1, dim))
    sample_points *= r  # Scale by random radius
    if noise > 0:
        noise_points = np.random.normal(0, 1, sample_points.shape) * noise * r
        sample_points += noise_points
    points[:n, :] = sample_points
    return points


def sample_rectangle(n, max_n=seq_length, dim=2, noise=0.05):
    points = np.full((max_n, dim), -1, dtype=np.float32)
    sample_points = np.random.uniform(-1, 1, (n, dim))
    sample_points /= np.linalg.norm(sample_points, axis=1, keepdims=True, ord=np.inf)
    r = np.random.uniform(0.5, 1, size = (1, dim))
    sample_points *= r  # Scale by random radius
    if noise > 0:
        noise_points = np.random.normal(0, 1, sample_points.shape) * noise * r
        sample_points += noise_points
    points[:n, :] = sample_points

    return points



dataset_size = 10000
X_sphere = np.zeros((dataset_size, seq_length, dim), dtype=np.float32)
X_square = np.zeros((dataset_size, seq_length, dim), dtype=np.float32)

for iter in range(dataset_size):
    n = seq_length
    X_sphere[iter, :, :] = sample_circle(n, seq_length, dim, noise)
    X_square[iter, :, :] = sample_circle(n, seq_length, dim, noise)

X = np.concatenate([X_sphere, X_square], axis=0)
y = np.concatenate(
    [
        np.zeros((dataset_size, 1), dtype=np.float32),
        np.ones((dataset_size, 1), dtype=np.float32),
    ],
    axis=0,
)

np.random.seed(42)
shuffle_indices = np.random.permutation(X.shape[0])
X = X[shuffle_indices]
y = y[shuffle_indices]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 12))
for i in range(5):
    ax.scatter(
        X_train[i, :, 0],
        X_train[i, :, 1],
        c="blue" if y_train[i] == 0 else "red",
        label="Circle" if y_train[i] == 0 else "Square",
    )
    

In [ ]:
from src.model.components import (
    MultiHeadAttentionBlock,
    MultiHeadAttentionStack,
    SelfAttentionBlock,
    SelfAttentionStack,
    PoolingAttentionBlock,
    point_transformer,
    GenerateMask,
    GetSequenceLength,
    DecoderQueries,
    GenerateDecoderMask,
    MLP,
    MaskOutput,
    DecoderPoints,
)

input_dim = dim
feature_dim = 8
latent_dim = 32
num_seeds = latent_dim // feature_dim
dropout_rate = 0.05
num_heads = 8
regularizer = keras.regularizers.l2(1e-5)

input = keras.layers.Input(shape=(seq_length, input_dim), name="input")

input_embedding = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="pixel_embedding",
    activation="relu",
    regularizer=regularizer,
)(input)

pixel_self_attention = SelfAttentionStack(
    num_heads=num_heads,
    key_dim=feature_dim,
    stack_size=3,
    name="pixel_self_attention",
    regularizer=regularizer,
)(input_embedding)

pooling = PoolingAttentionBlock(
    key_dim=feature_dim,
    num_seeds=num_seeds,
    num_heads=num_heads,
    name="pixel_pooling",
)(pixel_self_attention)


pooling_self_attention = SelfAttentionBlock(
    num_heads=num_heads,
    key_dim=feature_dim,
    name="pooling_self_attention",
    regularizer=regularizer,
)(pooling)

latent_space = MLP(
    num_layers=2,
    output_dim=feature_dim,
    name="latent_space",
    activation="linear",
    regularizer=regularizer,
    dropout_rate=dropout_rate,
)(pooling_self_attention)


decoder = MLP(
    num_layers=4,
    output_dim=seq_length * input_dim // num_seeds,
    name="decoder",
    activation="linear",
    regularizer=regularizer,
    dropout_rate=dropout_rate,
)(latent_space)

decoded_output = keras.layers.Reshape(
    (seq_length, input_dim), name="decoded_output"
)(decoder)

"""
seqlen_output = keras.layers.Concatenate(name="seqlen_output")(
    [
        keras.layers.Flatten(name="seqlen_flatten")(pixel_seqlen),
        keras.layers.Flatten(name="decoder_seqlen_flatten")(decoder_seqlen),
    ]
)"""

SetAE = keras.Model(
    inputs=input,
    outputs=[decoded_output],
    name="SetAutoEncoder",
)

In [ ]:
from src.training import (
    SplitMSE,
    ChamferDistanceMasked,
    EmbeddingSpaceSpreading,
    ChamferDistance,
    EmbeddingSpaceSpreading,
    EarthMoversDistanceLoss,
)

SetAE.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        "decoded_output": EarthMoversDistanceLoss(),
    },
    #loss_weights={"decoded_output": 1.0, "latent_output": 0.1},
)

SetAE.summary()
SetAE.fit(
    X,
    X,#[X, np.zeros((X.shape[0], latent_dim), dtype=np.float32)],
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    ],
)

In [ ]:
event = 3
sample = X_sphere[event : event + 1]
decoded_output = SetAE.predict(sample)
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(sample[0, :, 0], sample[0, :, 1], label="Input", color="blue")
ax.scatter(
    decoded_output[0, :, 0],
    decoded_output[0, :, 1],
    label="Decoded Output",
    color="red",
)
ax.set_title("Input vs Decoded Output")
ax.set_xlabel("X-axis")
ax.set_ylabel("Y-axis")
ax.legend()
plt.show()

In [ ]:
event = 0
sample = X_square[event : event + 1]
decoded_output = SetAE.predict(sample)
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(sample[0, :, 0], sample[0, :, 1], label="Input", color="blue")
ax.scatter(
    decoded_output[0, :, 0],
    decoded_output[0, :, 1],
    label="Decoded Output",
    color="red",
)
ax.set_title("Input vs Decoded Output")
ax.set_xlabel("X-axis")
ax.set_ylabel("Y-axis")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, figsize=(10, 10))
event_1 = 0
event_2 = 3
output_1 = SetAE.predict(X_train[event_1 : event_1 + 1])
output_2 = SetAE.predict(X_train[event_2 : event_2 + 1])
ax.scatter(
    output_1[0, :, 0],
    output_1[0, :, 1],
    label="Event 1",
    color="blue",
)
ax.scatter(
    output_2[0, :, 0],
    output_2[0, :, 1],
    label="Event 2",
    color="red",
)
ax.set_title("Latent Space Representation of Two Events")
ax.set_xlabel("X-axis")
ax.set_ylabel("Y-axis")
ax.legend()
plt.show()

In [ ]:
from src.model.components import (
    MultiHeadAttentionStack,
    SelfAttentionStack,
    PoolingAttentionBlock,
    GenerateMask,
    MLP,
    GetSequenceLength,
    DecoderQueries,
    point_transformer
)


input_dim = dim
feature_dim = 8
latent_dim = 32
num_seeds = latent_dim
dropout_rate = 0.0
num_heads = 8
regularizer = keras.regularizers.l2(1e-5)
dropout_rate = 0.0


pixel_input = keras.layers.Input(shape=(seq_length, input_dim), name="pixel_input")
pixel_mask = GenerateMask(name="pixel_mask")(pixel_input)
pixel_seqlen = GetSequenceLength(name="pixel_seqlen")(pixel_mask)

pixel_embedding = point_transformer(
    dim = feature_dim,
)(pixel_input,pixel_input)

pixel_self_attention = SelfAttentionStack(
    num_heads=num_heads,
    key_dim=feature_dim,
    stack_size=3,
    name="pixel_self_attention",
    regularizer=regularizer,
    dropout_rate=dropout_rate,
)(pixel_embedding)

pixel_pooling = PoolingAttentionBlock(
    key_dim=feature_dim,
    num_seeds=num_seeds,
    num_heads=num_heads,
    name="pixel_pooling",
    regularizer=regularizer,
    dropout_rate=dropout_rate,
)(pixel_self_attention)

pixel_latent_space = MLP(
    num_layers=4,
    output_dim=1,
    name="pixel_latent_space",
    activation="linear",
    regularizer=regularizer,
    dropout_rate=dropout_rate,
)(pixel_pooling)

pixel_latent_output = keras.layers.Flatten(name="pixel_latent_output")(
    pixel_latent_space
)

output = MLP(
    num_layers=4,
    output_dim=1,
    name="pixel_latent_output_mlp",
    activation="sigmoid",
    regularizer=regularizer,
    dropout_rate=dropout_rate,
)(pixel_latent_output)

transformer_model = keras.Model(
    inputs=pixel_input,
    outputs=output,
    name="ClassificationModel",
)
transformer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy()],
)

In [ ]:
transformer_model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
    ],
    class_weight={label: np.mean(y_train == label) for label in np.unique(y_train)},
    shuffle=True,
)

In [ ]:
test_prediction = transformer_model.predict(X_test)

In [ ]:
from sklearn.metrics import roc_curve, auc
fpr, tpr, thresholds = roc_curve(y_test, test_prediction, pos_label=1)
roc_auc = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(10, 10))
ax.plot(fpr, tpr, color="blue", label=f"ROC curve (area = {roc_auc:.2f})")
ax.plot([0, 1], [0, 1], color="red", linestyle="--", label="Random Guess")
ax.set_title("ROC Curve")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
plt.show()

In [ ]:
assign_fail = np.where((test_predictions > 0.5) != y_test)[0]



In [ ]:
y_test[assign_fail]

In [ ]:
plt.scatter(
    X_test[assign_fail, :, 0].flatten(),
    X_test[assign_fail, :, 1].flatten(),
    cmap="coolwarm",
    alpha=0.5,
)